# 07 real PX4 flight logs (no ground truth)

Design decisions:
- Source: public logs hosted by PX4 Flight Review (licensed CC-BY PX4), listed through its database endpoint and downloaded individually. No labels exist for these flights; they are treated as an unlabelled real-world population, not as a nominal class.
- Selection is fixed before any result is seen: multirotor airframes, 60 to 400 s of recorded flight, no logged errors, newest first, up to `MAX_LOGS` logs. The selection index is saved next to the logs.
- Features are extracted with the same `sih_features.py` as every other log; logs that lack a required topic are skipped and counted.
- Experiment E6 applies the pipeline trained on simulation only (40 training and 20 calibration flights per family, seed 0, as in the live-log experiment) and reports the verdict distribution, operational abstention, the gate and gap-rule withheld fractions, confidence, and alert-episode frequency, overall and by firmware release. Nothing is tuned on these logs.
- Implementation lives in `src/sih_features.py` and `src/sih_model.py`; this notebook runs them and presents the result files.

Result files: `data/real/px4_review/index.csv` (gitignored logs), `data/features/features_real/` (gitignored), `reports/v4/e6_real_flight_verdicts.csv`, `reports/v4/e6_real_summary.csv`, `reports/v4/e6_real_by_version.csv`. Manuscript: Section 6.7 (real logs), Table 10.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
# ---- selection and download (fixed before any result is seen) ----
import json, time, gzip, urllib.request, urllib.error
import pandas as pd
MAX_LOGS, MIN_DUR, MAX_DUR = 120, 60.0, 400.0
MULTIROTOR = ("quadrotor", "hexarotor", "octorotor", "multirotor", "tricopter", "coaxial")
DBINFO = "https://review.px4.io/dbinfo"
DOWNLOAD = "https://review.px4.io/download?log="
REAL_DIR = P.DATA / "real" / "px4_review"; REAL_DIR.mkdir(parents=True, exist_ok=True)

def fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "uav-gnss-triage/1.0", "Accept-Encoding": "gzip, identity"})
    with urllib.request.urlopen(req, timeout=180) as r:
        raw = r.read()
    if raw[:2] == b"\x1f\x8b":
        raw = gzip.decompress(raw)
    return json.loads(raw.decode("utf-8"))

def pick(d, *keys, default=None):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return default

entries = fetch_json(DBINFO)
print("entries listed:", len(entries)); print("fields:", sorted(entries[0].keys()))
rows = []
for e in entries:
    dur = pick(e, "duration_s", "log_duration", "duration", default=None)
    try:
        dur = float(dur)
    except (TypeError, ValueError):
        continue
    mav = str(pick(e, "mav_type", "type", default="")).lower()
    errs = pick(e, "num_logged_errors", "errors", default=0)
    try:
        errs = int(errs)
    except (TypeError, ValueError):
        errs = 0
    if any(m in mav for m in MULTIROTOR) and MIN_DUR <= dur <= MAX_DUR and errs == 0:
        rows.append({"log_id": pick(e, "log_id", "id"), "log_date": pick(e, "log_date", "date", default=""), "duration_s": dur,
                     "mav_type": pick(e, "mav_type", "type", default=""), "ver_sw_release": pick(e, "ver_sw_release", default=""),
                     "num_logged_errors": errs, "flight_modes": str(pick(e, "flight_modes", default=""))})
sel = pd.DataFrame(rows).sort_values("log_date", ascending=False).head(MAX_LOGS).reset_index(drop=True)
sel.to_csv(REAL_DIR / "index.csv", index=False)
print(f"selected {len(sel)} of {len(rows)} eligible logs (multirotor, {MIN_DUR:.0f}-{MAX_DUR:.0f} s, no logged errors, newest first)")
ok = fail = 0
for i, r in sel.iterrows():
    target = REAL_DIR / f"{r.log_id}.ulg"
    if target.exists() and target.stat().st_size > 0:
        ok += 1; continue
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(DOWNLOAD + str(r.log_id), str(target)); ok += 1; break
        except Exception as ex:
            time.sleep(2 + 3 * attempt)
            if attempt == 2:
                fail += 1; print("failed:", r.log_id, type(ex).__name__)
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(sel)} downloaded")
print(f"downloaded: {ok} | failed: {fail} | on disk: {len(list(REAL_DIR.glob('*.ulg')))}")

In [ ]:
# ---- feature extraction with the shared extractor ----
subprocess.run(["pip", "install", "-q", "pyulog", "pandas"], check=True)
FEAT_REAL = P.FEATURES / "features_real"
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_features.py"), "--real_dir", str(REAL_DIR), "--out", str(FEAT_REAL)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = [l for l in proc.stdout]
print("exit code:", proc.wait())
skipped = [l for l in lines if l.startswith("SKIP")]
print("".join(l for l in lines if l.startswith("wrote") or l.startswith("SKIP")))
print("skipped logs:", len(skipped))


In [ ]:
# ---- E6: frozen pipeline on the real logs ----
subprocess.run(["pip", "install", "-q", "xgboost", "scikit-learn", "scipy", "pandas"], check=True)
RUN, FEAT = "v4", "features_v3"
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_model.py"), "--features", str(P.FEATURES / FEAT), "--out", str(P.REPORTS / RUN),
                         "--only", "e6", "--real_features", str(FEAT_REAL)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())


In [ ]:
# ---- presentation of the committed result files ----
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
out = P.REPORTS / RUN
summary = pd.read_csv(out / "e6_real_summary.csv").T.rename(columns={0: "value"})
display(summary)
if (out / "e6_real_by_version.csv").exists():
    display(pd.read_csv(out / "e6_real_by_version.csv"))
vr = pd.read_csv(out / "e6_real_flight_verdicts.csv")
display(vr[["flight_id", "pred", "conf", "set_a0.10", "coarse_pred", "coarse_set_a0.10", "abstain_fine", "gate_mahal", "gate_gaprule", "alert_episodes",
            "window_frac_nominal", "window_frac_spoof", "window_frac_gps_degrade", "window_frac_sensor_fault"] + [c for c in ("duration_s", "ver_sw_release") if c in vr.columns]]
        .sort_values("conf").head(25).round(3))
print("verdict counts:", vr["pred"].value_counts().to_dict())
print("confidence quantiles:", vr["conf"].quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(3).to_dict())
